# 🎨 ComfyUI Portable Cloud Studio (Colab T4 Free)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SudhirDevOps1/t4-gpu-llm-api/blob/main/colab/05_comfyui_cloud_studio.ipynb)
[![Lightning AI](https://img.shields.io/badge/Lightning_AI-Open_Studio-792EE5?style=for-the-badge&logo=lightning)](https://lightning.ai/new?repo_url=https://github.com/SudhirDevOps1/t4-gpu-llm-api)
[![GitHub Store](https://img.shields.io/badge/Notebook_Store-Obsidian_Hub-10b981?style=for-the-badge&logo=github)](https://github.com/SudhirDevOps1/t4-gpu-llm-api)

> **⚡ Architecture Highlights:**
> - 🚀 **1-Click ComfyUI + ComfyUI-Manager:** Visual node-based workflow editor in the cloud.
> - 🛡️ **Zero-Leak Security:** Checkpoint models downloaded to separate storage outside git.
> - 🌐 **High-Speed Cloudflare Tunnel:** Persistent HTTPS connection without local port forwarding.
> - 💻 **VRAM Optimization:** Configured with --fp16-vae --force-fp16 for Colab T4 (15.3 GB).


In [ ]:
#@title 🚀 1. Hardware Check & Zero-Leak Session Fingerprint
import os, sys, time, uuid, socket, hashlib

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
raw_fp = f"colab-comfy-{socket.gethostname()}-{uuid.uuid4()}-{time.time()}"
INSTANCE_ID = "colab-comfy-" + hashlib.sha256(raw_fp.encode()).hexdigest()[:8]
print(f"🆔 Unique Instance Fingerprint: {INSTANCE_ID}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv


In [ ]:
#@title 📦 2. Clone ComfyUI, ComfyUI-Manager & Install Requirements
!git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager
!pip install -q -r /content/ComfyUI/requirements.txt
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
print("✅ ComfyUI and ComfyUI-Manager installed successfully.")


In [ ]:
#@title 🌐 3. Launch ComfyUI Server & Expose Cloudflare Tunnel
import subprocess, time, re, os
from IPython.display import clear_output, Markdown, display

# Kill prior instances
subprocess.run(["pkill", "-9", "cloudflared"], check=False)
time.sleep(1)

log_path = "/tmp/comfy_tunnel.log"
if os.path.exists(log_path): os.remove(log_path)

# 1. Start ComfyUI in background
comfy_proc = subprocess.Popen([
    "python", "/content/ComfyUI/main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--fp16-vae",
    "--preview-method", "auto"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)

# 2. Expose Cloudflare Tunnel to port 8188
log_file = open(log_path, "w")
tunnel_proc = subprocess.Popen([
    "cloudflared", "tunnel", "--url", "http://127.0.0.1:8188"
], stdout=log_file, stderr=subprocess.STDOUT)

public_url = None
for _ in range(35):
    time.sleep(1)
    if os.path.exists(log_path):
        with open(log_path, "r") as f:
            match = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", f.read())
            if match:
                public_url = match.group(0)
                break

clear_output()
if public_url:
    display(Markdown(f"""
### 🎨 **ComfyUI Portable Cloud Studio is Online!**

| Metric | Setting |
| :--- | :--- |
| **Public Web UI URL** | [{public_url}]({public_url}) |
| **Internal Port** | 8188 (Bound to 127.0.0.1) |
| **ComfyUI-Manager** | Active (1-Click Custom Node & Model Install) |
| **Session ID** | {INSTANCE_ID} |
| **GPU Optimization** | --fp16-vae (T4 GPU Memory Protected) |

👉 **Click the link above to open your ComfyUI node canvas!**
"""))
else:
    print("❌ Tunnel failed to launch. Check /tmp/comfy_tunnel.log for details.")
